# Case 1: 2D Channel Facies Geomodeling

This notebook demonstrates both unconditional and conditional diffusion for 2D channel facies modeling.

In [ ]:
import sys
sys.path.insert(0, '..')

import json
import torch
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from diffsim.models import Unet
from diffsim.core.diffusion import Diffusion
from diffsim.core.network import Network
from diffsim.data.dataset import InpaintDatasetCase1

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

def set_seed(seed=42):
    """Set random seed for reproducibility."""
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    print(f"Random seed set to {seed}")

def set_device(data):
    """Move tensor to device."""
    if isinstance(data, dict):
        return {k: set_device(v) for k, v in data.items()}
    elif isinstance(data, torch.Tensor):
        return data.to(device)
    return data

In [ ]:
# Load config
with open('../configs/case1_geomodeling.json') as f:
    config = json.load(f)
print(f"Loaded config for: {config['name']}")

## Section 1: Unconditional Generation

In [ ]:
# Build unconditional model
uncond = config['unconditional']
model = Unet(
    dim=uncond['dim'],
    channels=uncond['channels'],
    dim_mults=tuple(uncond['dim_mults'])
)
model.to(device)

# Load checkpoint
ckpt_path = Path('..') / config['checkpoints']['unconditional']
if ckpt_path.exists():
    model.load_state_dict(torch.load(ckpt_path, map_location=device))
    print(f"Loaded checkpoint from {ckpt_path}")
else:
    print(f"Checkpoint not found at {ckpt_path}. Using random weights.")

In [ ]:
# Initialize diffusion
diffusion = Diffusion(
    timesteps=uncond['timesteps'],
    beta_schedule=uncond['beta_schedule']
)

# Generate samples using DDPM
set_seed(42)
model.eval()
with torch.no_grad():
    samples_ddpm = diffusion.sample(
        model,
        image_size=config['image_size'],
        batch_size=16,
        channels=uncond['channels']
    )

In [ ]:
# Visualize DDPM generated samples
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for i, ax in enumerate(axes.flat):
    if i < samples_ddpm.shape[0]:
        img = samples_ddpm[i, 0].cpu().numpy()
        ax.imshow(img, cmap='viridis')
    ax.axis('off')
plt.suptitle('DDPM Generated Samples (1500 steps)', fontsize=16)
plt.tight_layout()
plt.show()

### DDIM Sampling (Faster)

In [ ]:
# Generate samples using DDIM (faster, 50 steps instead of 1500)
set_seed(42)
model.eval()
with torch.no_grad():
    samples_ddim = diffusion.sample_ddim(
        model,
        image_size=config['image_size'],
        batch_size=16,
        channels=uncond['channels'],
        ddim_steps=50,
        eta=0.0  # deterministic
    )

In [ ]:
# Visualize DDIM generated samples
fig, axes = plt.subplots(4, 4, figsize=(12, 12))
for i, ax in enumerate(axes.flat):
    if i < samples_ddim.shape[0]:
        img = samples_ddim[i, 0].cpu().numpy()
        ax.imshow(img, cmap='viridis')
    ax.axis('off')
plt.suptitle('DDIM Generated Samples (50 steps)', fontsize=16)
plt.tight_layout()
plt.show()

In [ ]:
# Compare DDPM vs DDIM
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for i in range(4):
    axes[0, i].imshow(samples_ddpm[i, 0].cpu().numpy(), cmap='viridis')
    axes[0, i].set_title('DDPM' if i == 0 else '')
    axes[0, i].axis('off')
    
    axes[1, i].imshow(samples_ddim[i, 0].cpu().numpy(), cmap='viridis')
    axes[1, i].set_title('DDIM' if i == 0 else '')
    axes[1, i].axis('off')
plt.suptitle('DDPM (1500 steps) vs DDIM (50 steps)', fontsize=16)
plt.tight_layout()
plt.show()

## Section 2: Conditional Generation (Inpainting with Sparse Well Conditions)

The conditional model takes sparse well observations as conditioning input.

In [ ]:
# Build conditional network
cond = config['conditional']
unet_config = {
    "image_size": config['image_size'],
    "in_channel": cond['in_channel'],
    "out_channel": cond['out_channel'],
    "inner_channel": cond['inner_channel'],
    "channel_mults": cond['channel_mults'],
    "attn_res": cond['attn_res'],
    "num_head_channels": cond['num_head_channels'],
    "res_blocks": cond['res_blocks'],
    "dropout": cond['dropout'],
}

network = Network(
    unet=unet_config,
    beta_schedule=cond['beta_schedule'],
    module_name='guided_diffusion'
)
network.to(device)

# Load checkpoint with strict=False (ignore buffer mismatches if any)
ckpt_path = Path('..') / config['checkpoints']['conditional']
if ckpt_path.exists():
    network.load_state_dict(torch.load(ckpt_path, map_location=device, weights_only=False), strict=False)
    print(f"Loaded checkpoint from {ckpt_path}")
else:
    print(f"Checkpoint not found at {ckpt_path}. Using random weights.")

# Setup noise schedule for test (use 200 steps for faster sampling)
# This will register the buffers with the correct size for inference
test_schedule = {
    "schedule": cond['beta_schedule']['test']['schedule'],
    "n_timestep": 200,  # Reduced from 1500 for faster sampling
    "linear_start": cond['beta_schedule']['test']['linear_start'],
    "linear_end": cond['beta_schedule']['test']['linear_end']
}
network.beta_schedule['test'] = test_schedule
network.set_new_noise_schedule(device=torch.device(device), phase='test')

print(f"Using {network.num_timesteps} timesteps for sampling")

### Load Dataset with InpaintDataset

In [ ]:
# Define data paths from config (support both old and new config structure)
cond_data = config.get('conditional', {}).get('data', config.get('data', {}))
images_path = cond_data.get('test_image', cond_data.get('test_image_path', ''))
masks_path = cond_data.get('test_mask', cond_data.get('test_mask_path', ''))
data_root = (images_path, masks_path)

# Mask configuration
mask_config = {
    'mask_mode': 'file'  # Use mask files from the masks directory
}

# Load dataset
dataset = InpaintDatasetCase1(data_root, mask_config=mask_config, data_len=-1, image_size=[64, 64])
print(f"Dataset size: {len(dataset)}")

### Visualize One Sample

In [ ]:
# Plot one sample from the dataset
plt.figure(figsize=(15, 3))
inum = 265

plt.subplot(151)
plt.imshow(dataset[inum]['gt_image'].cpu().numpy().reshape(64, 64))
plt.colorbar()
plt.title('Test image')

plt.subplot(152)
plt.imshow(dataset[inum]['yt_image'].cpu().numpy().reshape(64, 64))
plt.colorbar()
plt.title('T images')

plt.subplot(153)
plt.imshow(dataset[inum]['mask_image'].cpu().numpy().reshape(64, 64))
plt.colorbar()
plt.title('Mask images')

plt.subplot(154)
plt.imshow(dataset[inum]['mask'].cpu().numpy().reshape(64, 64), 'gray')
plt.colorbar()
plt.title('Mask')

plt.subplot(155)
plt.imshow(dataset[inum]['cond_image'][3])
plt.colorbar()
plt.title('Conditional images')

plt.tight_layout()
plt.show()

### Generate Multiple Realizations for Test Samples

In [ ]:
# Test indices for generation
numlist = [1421, 290, 1988, 176, 265, 0, 2302, 1081]
num_realizations = 10

print(f"Generating {num_realizations} realizations for {len(numlist)} test samples...")

In [ ]:
# Generate realizations for each test sample using batch forward
set_seed(42)
network.eval()

# Store all outputs: [num_samples, num_realizations, H, W]
total_output = np.zeros((len(numlist), num_realizations, 64, 64))

for idx, inum in enumerate(numlist):
    print(f"Processing sample {idx+1}/{len(numlist)} (index {inum})...")
    
    # Get data from dataset
    img = dataset[inum]['gt_image']
    mask = dataset[inum]['mask']
    cond_image = dataset[inum]['cond_image']
    yt_image = dataset[inum]['yt_image']
    
    # Prepare batched inputs by repeating for num_realizations
    cond_input = torch.from_numpy(cond_image).unsqueeze(0).repeat(num_realizations, 1, 1, 1).to(device)
    gt_image_batch = img.unsqueeze(0).repeat(num_realizations, 1, 1, 1).to(device)
    mask_input = mask.unsqueeze(0).repeat(num_realizations, 1, 1, 1).to(device)
    # Generate fresh noise for each realization
    yt_input = img * (1. - mask) + mask * torch.randn(num_realizations, 1, 64, 64)
    yt_input = yt_input.to(device)
    
    with torch.no_grad():
        output, visuals = network.restoration(
            y_cond=cond_input, 
            y_t=yt_input,
            y_0=gt_image_batch, 
            mask=mask_input, 
            sample_num=10
        )
    
    total_output[idx, :, :, :] = output.cpu().numpy().reshape(num_realizations, 64, 64)

print("Generation complete!")
print(f"Output shape: {total_output.shape}")

In [ ]:
# Compute mean probability for each facies across realizations
# Facies values (normalized to [-1, 1]): Sand ~1, Bank ~0, Mud ~-1

# Define thresholds for facies classification
sand_threshold = 0.5    # Values > 0.5 are sand
mud_threshold = -0.5    # Values < -0.5 are mud
# Bank is in between

# Compute binary masks for each facies across all realizations
sand_masks = (total_output > sand_threshold).astype(np.float32)  # [num_samples, num_realizations, H, W]
mud_masks = (total_output < mud_threshold).astype(np.float32)
bank_masks = ((total_output >= mud_threshold) & (total_output <= sand_threshold)).astype(np.float32)

# Compute mean probability (frequency) for each facies
mean_sand = np.mean(sand_masks, axis=1)  # [num_samples, H, W]
mean_bank = np.mean(bank_masks, axis=1)
mean_mud = np.mean(mud_masks, axis=1)

print(f"Mean sand shape: {mean_sand.shape}")
print(f"Mean bank shape: {mean_bank.shape}")
print(f"Mean mud shape: {mean_mud.shape}")

### Plot Results: Test Images, Realizations, Mean, and Variance

In [ ]:
# Create output directory
dir_name = '../results/case1'
Path(dir_name).mkdir(parents=True, exist_ok=True)

import matplotlib.cm as cm
import matplotlib.patches as mpatches

fig, ax = plt.subplots(8, 11, sharex='col', sharey='row')
fig.set_size_inches(11, 8, forward=True)

for i in range(8):
    inum = numlist[i]
    re_mask = 1 - dataset[inum]['mask'].cpu().numpy().reshape(64, 64).astype(np.float32)
    re_mask[re_mask == 0] = np.nan
    gt = dataset[inum]['gt_image'].cpu().numpy().reshape(64, 64)
    gt[np.isnan(re_mask)] = np.nan

    ax[i, 0].imshow(dataset[inum]['gt_image'].cpu().numpy().reshape(64, 64))
    ax[i, 1].imshow(gt)
    # Scatter plot on top of the facies image to enlarge points
    facies = gt.copy()
    y, x = np.where(~np.isnan(facies))
    ax[i, 1].scatter(x, y, c=facies[~np.isnan(facies)], cmap='viridis', s=10, marker='s', vmin=-1, vmax=1)

    for j in range(6):
        ax[i, j+2].imshow(total_output[i, j, :, :])
    
    # Column 8: Channel Sand (mean probability)
    ax[i, 8].imshow(mean_sand[i], cmap='jet', vmin=0, vmax=1)
    
    # Column 9: Channel Bank (mean probability)
    ax[i, 9].imshow(mean_bank[i], cmap='jet', vmin=0, vmax=1)
    
    # Column 10: Inter-channel Mud (mean probability)
    h = ax[i, 10].imshow(mean_mud[i], cmap='jet', vmin=0, vmax=1)

ax[0, 0].set_title('Test faceis\nmodel', fontsize=10)
ax[0, 1].set_title('Conditioning\nfacies', fontsize=10)
ax[0, 5].set_title('Realizations')
ax[0, -1].set_title('Inter-channel\nMud', fontsize=10)
ax[0, -2].set_title('Channel\nBank', fontsize=10)
ax[0, -3].set_title('Channel\nSand', fontsize=10)

# Hide labels but keep ticks
for i in range(8):
    for j in range(11):
        ax[i, j].tick_params(labelbottom=False, labelleft=False)
        
plt.tight_layout()
right = 0.89
plt.subplots_adjust(left=0.02, bottom=0.05, right=right, top=0.95, wspace=0.15, hspace=0.15)

# Create custom axes for the colorbars on the right of the plot
cbaxes_mean = fig.add_axes([right+0.01, 0.1, 0.02, 0.4])

# Add colorbars
cbar_mean = fig.colorbar(h, ax=ax[:, -2], orientation='vertical', fraction=0.046, pad=0.04, cax=cbaxes_mean)

# Set labels for the colorbars
cbar_mean.set_label('Mean of facies')

# Sample colors from the 'viridis' colormap
cmap = cm.get_cmap('viridis')
channel_mud_color = cmap(0.0)   # -1: Channel Mud
channel_bank_color = cmap(0.5)  # 0: Channel Bank
channel_sand_color = cmap(1.0)  # 1: Channel Sand

# Create custom legend handles with square patches
channel_mud = mpatches.Patch(color=channel_mud_color, label='Inter-channel Mud')
channel_bank = mpatches.Patch(color=channel_bank_color, label='Channel Bank')
channel_sand = mpatches.Patch(color=channel_sand_color, label='Channel Sand')

# Add the legend to the figure using fig.legend
legend = fig.legend(handles=[channel_mud, channel_bank, channel_sand], loc='lower center', bbox_to_anchor=(0.5, -0.02), ncol=3, frameon=False)

plt.savefig(dir_name + '/FigureCase1.png', dpi=300)
plt.show()

print(f"Figure saved to {dir_name}/FigureCase1.png")

### DDIM Sampling for Conditional Generation (Faster)

In [ ]:
# Generate realizations using DDIM (faster, 50 steps instead of 200)
set_seed(42)
network.eval()

# Store all outputs: [num_samples, num_realizations, H, W]
total_output_ddim = np.zeros((len(numlist), num_realizations, 64, 64))

print(f"Generating {num_realizations} realizations for {len(numlist)} test samples using DDIM...")

for idx, inum in enumerate(numlist):
    print(f"Processing sample {idx+1}/{len(numlist)} (index {inum})...")
    
    # Get data from dataset
    img = dataset[inum]['gt_image']
    mask = dataset[inum]['mask']
    cond_image = dataset[inum]['cond_image']
    
    # Prepare batched inputs by repeating for num_realizations
    cond_input = torch.from_numpy(cond_image).unsqueeze(0).repeat(num_realizations, 1, 1, 1).to(device)
    gt_image_batch = img.unsqueeze(0).repeat(num_realizations, 1, 1, 1).to(device)
    mask_input = mask.unsqueeze(0).repeat(num_realizations, 1, 1, 1).to(device)
    # Generate fresh noise for each realization
    yt_input = img * (1. - mask) + mask * torch.randn(num_realizations, 1, 64, 64)
    yt_input = yt_input.to(device)
    
    with torch.no_grad():
        output, visuals = network.restoration_ddim(
            y_cond=cond_input, 
            y_t=yt_input,
            y_0=gt_image_batch, 
            mask=mask_input, 
            ddim_steps=50,  # Much faster than 200 DDPM steps
            eta=0.0,        # Deterministic (set eta>0 for stochasticity)
            sample_num=8
        )
    
    total_output_ddim[idx, :, :, :] = output.cpu().numpy().reshape(num_realizations, 64, 64)

print("DDIM Generation complete!")
print(f"Output shape: {total_output_ddim.shape}")

In [ ]:
# Compute mean probability for each facies across DDIM realizations
sand_masks_ddim = (total_output_ddim > sand_threshold).astype(np.float32)
mud_masks_ddim = (total_output_ddim < mud_threshold).astype(np.float32)
bank_masks_ddim = ((total_output_ddim >= mud_threshold) & (total_output_ddim <= sand_threshold)).astype(np.float32)

mean_sand_ddim = np.mean(sand_masks_ddim, axis=1)
mean_bank_ddim = np.mean(bank_masks_ddim, axis=1)
mean_mud_ddim = np.mean(mud_masks_ddim, axis=1)

print(f"DDIM Mean sand shape: {mean_sand_ddim.shape}")

### Plot DDIM Results

In [ ]:
# Plot DDIM results: Test Images, Realizations, Mean Facies Probabilities
fig, ax = plt.subplots(8, 11, sharex='col', sharey='row')
fig.set_size_inches(11, 8, forward=True)

for i in range(8):
    inum = numlist[i]
    re_mask = 1 - dataset[inum]['mask'].cpu().numpy().reshape(64, 64).astype(np.float32)
    re_mask[re_mask == 0] = np.nan
    gt = dataset[inum]['gt_image'].cpu().numpy().reshape(64, 64)
    gt_cond = gt.copy()
    gt_cond[np.isnan(re_mask)] = np.nan

    ax[i, 0].imshow(gt)
    ax[i, 1].imshow(gt_cond)
    y, x = np.where(~np.isnan(gt_cond))
    ax[i, 1].scatter(x, y, c=gt_cond[~np.isnan(gt_cond)], cmap='viridis', s=10, marker='s', vmin=-1, vmax=1)

    for j in range(6):
        ax[i, j+2].imshow(total_output_ddim[i, j, :, :])
    
    ax[i, 8].imshow(mean_sand_ddim[i], cmap='jet', vmin=0, vmax=1)
    ax[i, 9].imshow(mean_bank_ddim[i], cmap='jet', vmin=0, vmax=1)
    h = ax[i, 10].imshow(mean_mud_ddim[i], cmap='jet', vmin=0, vmax=1)

ax[0, 0].set_title('Test facies\nmodel', fontsize=10)
ax[0, 1].set_title('Conditioning\nfacies', fontsize=10)
ax[0, 5].set_title('DDIM Realizations')
ax[0, -1].set_title('Inter-channel\nMud', fontsize=10)
ax[0, -2].set_title('Channel\nBank', fontsize=10)
ax[0, -3].set_title('Channel\nSand', fontsize=10)

for i in range(8):
    for j in range(11):
        ax[i, j].tick_params(labelbottom=False, labelleft=False)
        
plt.tight_layout()
right = 0.89
plt.subplots_adjust(left=0.02, bottom=0.05, right=right, top=0.95, wspace=0.15, hspace=0.15)

cbaxes_mean = fig.add_axes([right+0.01, 0.1, 0.02, 0.4])
cbar_mean = fig.colorbar(h, orientation='vertical', cax=cbaxes_mean)
cbar_mean.set_label('Mean of facies')

legend = fig.legend(handles=[channel_mud, channel_bank, channel_sand], loc='lower center', bbox_to_anchor=(0.5, -0.02), ncol=3, frameon=False)

plt.savefig(dir_name + '/FigureCase1_DDIM.png', dpi=300)
plt.show()

print(f"DDIM Figure saved to {dir_name}/FigureCase1_DDIM.png")